In [4]:
from util_funcs.cmi_utils import *
from pathlib import Path
# import kaggle_evaluation.cmi_inference_server

In [5]:
stringify_pthlib_pth = lambda pth: str(pth.resolve())

In [3]:

models_dir = Path("/kaggle/input/second-dset")
rf_pth,encoder_pth = [stringify_pthlib_pth(pth) for pth in models_dir.iterdir()]

target_col = "gesture"
common_extra_cols = ["row_id","sequence_counter"]
rotation_cols = [ 'rot_w', 'rot_x',
       'rot_y', 'rot_z']
accelerometer_cols = ['acc_x', 'acc_y', 'acc_z']

thermophile_columns = [f"thm_{i}" for i in range(1, 6)]
tof_cols = [f"tof_{i}_v{j}" for i in range(1, 6) for j in range(64)]
grouping_col = "sequence_id"
subject_col = "subject"
heat_col = "total_heat"
# doing the fe
cats_to_be = ["sex","handedness","adult_child"]

In [4]:
rf_model = reloading_pickles(rf_pth)

In [5]:
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    
    # Replace this function with your inference code.
    # You can return either a Pandas or Polars dataframe, though Polars is recommended.
    # Each prediction (except the very first) must be returned within 30 minutes of the batch features being provided.
    test_df = convert_df(sequence)
    test_demog_df = convert_df(demographics)
    encoder_pth='/kaggle/input/second-dset/lblencoder.pkl'
    rf_pth = '/kaggle/input/second-dset/second_rf.pkl'
    lablencoder = reloading_pickles(encoder_pth)
    rf_model = reloading_pickles(rf_pth)
    all_cols = list(rf_model.feature_names_in_)
    drop_cols(test_df,[*tof_cols,*common_extra_cols])
   
    test_demog_df = vanilla_categorical_procs(test_demog_df,cats_to_be)
    test_df = vanilla_quant_procs(test_df)
    all_numeric_cols = [*accelerometer_cols,*rotation_cols]
    penultimate_df = perform_grouped_analysis(test_df,grouping_col,thermophile_columns,all_numeric_cols,heat_col)
    penultimate_df = penultimate_df.reset_index()
    drop_cols(test_df,[*accelerometer_cols,*rotation_cols,*thermophile_columns,heat_col],True)
    cleaned_df = test_df
    merged_numeric_df = pd.merge(penultimate_df,cleaned_df,on=grouping_col)
    merged_numeric_df.drop_duplicates(inplace=True)
    complete_df = merged_numeric_df.merge(test_demog_df,on=subject_col)
    seq_ids = complete_df[grouping_col].values.tolist()
    drop_cols(complete_df,[subject_col,grouping_col],True)
    complete_df = complete_df[all_cols]
    complete_df = complete_df.fillna(0)
    results = rf_model.predict(complete_df)
    final_labels = lablencoder.inverse_transform(results)[-1]
    
    return final_labels

In [6]:
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
        )
    )